# Reporting (Stage 7)

Turn a populated store into the community products: **aggregate** statistics
(region/season + HEALPix), an aggregate `.rst` page, a standalone **Bokeh**
scatter, and a **download manifest** — all offline, with the object-store/Zenodo
uploads stubbed (`pab.report`). No per-matchup pages are generated.

## 1. A synthetic populated store

Seed a handful of matchups with BING fits (`bbp`/`chl`), Argo summaries, and a couple of artifact files so the manifest has something to point at.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
from pathlib import Path
from pab.db import Store
from pab.argo.summary import persist_summary

store = Store.open(":memory:")
rng = np.random.default_rng(7)
sites = [("N_Atl", 27.0, -46.0, "2025-02-18T20:00:00"),
         ("Eq_Pac", 4.0, -137.0, "2025-07-25T13:00:00"),
         ("S_Atl", -30.0, -10.0, "2025-05-10T11:00:00")]
art = Path("artifacts"); art.mkdir(exist_ok=True)
for k, (name, lat, lon, t) in enumerate(sites):
    for j in range(3):
        wmo, cyc = 7900000 + k, 5 + j
        bbp_argo = 10 ** rng.uniform(-3.2, -2.5)
        bbp_bing = bbp_argo * 1.3 * 10 ** rng.normal(0, 0.1)
        pid = persist_summary(store, wmo=wmo, cycle=cyc,
            summary={"mld": 30., "mld_method": "x", "bbp700": bbp_argo,
                     "chla": 10**rng.uniform(-1.2, 0.2), "n_points": 6},
            latitude=lat + rng.normal(0, 0.05), longitude=lon + rng.normal(0, 0.05), time=t)
        gid = f"G_{wmo}_{cyc}"; mid = f"{wmo}_{cyc}_{gid}"
        store.upsert("granules", {"granule_id": gid, "data_url": f"s3://b/{gid}.nc"})
        store.upsert("matchups", {"matchup_id": mid, "profile_id": pid,
                                  "granule_id": gid, "n_spectra": 1})
        store.upsert("matchup_pixels", {"matchup_id": mid, "ix": 2, "iy": 2,
                                        "rank": 1, "flagged": 0})
        fid = f"{mid}_2_2_ExpBPow"
        npz = art / f"{fid}.npz"; npz.write_bytes(b"chains")
        store.upsert("fits", {"fit_id": fid, "matchup_id": mid, "algorithm": "BING",
                              "model_pair": "ExpBPow", "chisq": float(rng.uniform(.8,1.5)),
                              "chains_path": str(npz), "pab_version": "0"})
        for q, v in (("bbp700", bbp_bing), ("chl", 10**rng.uniform(-1, 0.2))):
            store.upsert("fit_results", {"fit_id": fid, "quantity": f"BING_ExpBPow_{q}",
                                         "value": v, "value_lo": v*.9, "value_hi": v*1.1, "unit": ""})
print("matchups:", store.count("matchups"), "fits:", store.count("fits"))

matchups: 9 fits: 9


## 2. Aggregate statistics

Bin the per-matchup comparison by region and season, and over equal-area HEALPix cells.

In [2]:
from pab.metrics import compare
from pab.report import aggregate

df = compare.add_strata(compare.gather_matchups(store))
print("by region:")
print(aggregate.aggregate_by(df, "region")[["region","n","median_ratio","spearman"]].to_string(index=False))
print("\nHEALPix cells (~5 deg):")
hp = aggregate.aggregate_healpix(df, cell_size_deg=5.0)
print(hp[["hpix","lon","lat","n","median_ratio"]].to_string(index=False))

by region:


    region  n  median_ratio  spearman
subtropics  6      1.454232       1.0
   tropics  3      1.290301       1.0

HEALPix cells (~5 deg):


 hpix      lon        lat  n  median_ratio
  236  -45.000  24.624318  3      1.392573
  356 -135.000   4.780192  3      1.290301
  591   -5.625 -30.000000  3      1.515891


## 3. Aggregate `.rst` page (no per-matchup pages)

`build_site` writes a **fixed** set of pages regardless of matchup count.

In [3]:
from pab.report import rst
written = rst.build_site(store, "report_site")
print("pages:", sorted(written))
print("\n--- summary.rst (head) ---")
print("\n".join(written["summary"].read_text().splitlines()[:14]))

pages: ['aggregates', 'index', 'methods', 'summary']

--- summary.rst (head) ---
PAB matchup results


PACE ↔ BGC-Argo matchups: satellite vs. in-situ backscatter (``b_bp``) and chlorophyll, retrieved with BING. Built from ``pab_version`` ``0.0.dev0`` on 2026-06-24.

Coverage
--------

- **Matchups:** 9
- **Floats:** 3
- **BING fits:** 9

Headline comparison (b_bp 700 nm)


## 4. Standalone Bokeh scatter

Satellite-vs-float `b_bp`, log-log, with hover. It embeds as **standalone** HTML+JS (`bokeh.embed.components`, no server) — the build drops the script+div into a `.. raw:: html` block. We print the embed sizes here to keep the docs notebook static.

In [4]:
from pab.report import interactive

fig = interactive.comparison_scatter(df, title="BING vs Argo b_bp")
script, div = interactive.embed(fig)   # standalone HTML+JS, no server
print("embed script bytes:", len(script), "| div bytes:", len(div))
print("hover fields:", [c for c in interactive.HOVER_FIELDS if c in df.columns])
# In the built site this script+div is dropped into a `.. raw:: html` block
# (interactive.raw_html(fig)); it renders as an interactive scatter with
# hover + tap-to-artifact. Shown here as sizes to keep the notebook static.

embed script bytes: 11502 | div bytes: 101
hover fields: ['matchup_id', 'wmo', 'cycle', 'bbp_bing', 'bbp_argo', 'chl_bing']


## 5. Downloads + manifest (stubbed publish)

Export the summary table and build a download manifest (id → URL + checksum), 'uploading' artifacts via the local stub backend (no network).

In [5]:
from pab.report import publish
res = publish.publish_release(store, "release", base_url="https://store.example/pab")
print("exports:", {k: v.name for k, v in res["exports"].items()})
print("artifacts uploaded:", res["n_uploaded"])
print("manifest rows:", len(res["manifest"]))
print("example:", {k: res["manifest"][0][k] for k in ("matchup_id","kind","url","checksum")})

exports: {'summary_csv': 'matchup_summary.csv', 'summary_parquet': 'matchup_summary.parquet'}
artifacts uploaded: 9
manifest rows: 9
example: {'matchup_id': '7900000_5_G_7900000_5', 'kind': 'chains', 'url': 'file://local/7900000_5_G_7900000_5_2_2_ExpBPow.npz', 'checksum': '264ad4c6a1169d92'}
